# Protein interface analysis

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/davismerighi-max/interface-analysis/blob/main/notebooks/interface_analysis_colab.ipynb)

Upload a refined structure — `.pdb` or `.cif`, or just a PDB entry code — and this notebook
measures every interface in it: which residues are buried and by how much, which salt bridges,
hydrogen bonds, hydrophobic contacts, disulfides, metal bridges and stacking interactions hold
the subunits together, how well the two surfaces fit, and what the bound cofactors touch.
It then draws the figures and gives you everything as a zip.

**You do not need to know any programming.** Run the cells in order, from top to bottom
(`Shift`+`Enter`, or the ▶ button on each cell). Where a cell has boxes, fill them in first.

The symmetry, the number of chains and the number of different proteins are measured from
your model, never declared, so a simple dimer and an icosahedral capsid both work.

Source code and documentation: https://github.com/davismerighi-max/interface-analysis


## 1 · Set up

Installs the libraries and downloads the code. Takes about a minute. Run it once per session.


In [ ]:
#@title Install and fetch the code { display-mode: "form" }
import os, sys, subprocess, shutil, pathlib

print("installing libraries ...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "biopython", "pydssp", "scikit-image"], check=True)

REPO = "/content/interface-analysis"
if os.path.exists(REPO):
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "--depth", "1", "-q",
                "https://github.com/%s.git" % "davismerighi-max/interface-analysis", REPO], check=True)

WORK = "/content/work"
os.makedirs(WORK, exist_ok=True)
for f in pathlib.Path(REPO).glob("*.py"):
    shutil.copy(f, WORK)
os.chdir(WORK)
print("ready — working directory is", os.getcwd())

## 2 · Give it a structure

Either type a four-character PDB entry code, or leave it empty and upload your own file.
A refined model straight out of Phenix, Refmac, Coot or ChimeraX is fine.


In [ ]:
#@title Load the model { display-mode: "form" }
pdb_entry_code = "2TSC"  #@param {type:"string"}
#@markdown Leave the box empty to upload your own `.pdb` / `.cif` file instead.

import os, glob, urllib.request

MODEL = None
code_ = pdb_entry_code.strip().lower()
if code_:
    MODEL = code_ + ".pdb"
    url = "https://files.rcsb.org/download/" + MODEL
    print("downloading", url)
    urllib.request.urlretrieve(url, MODEL)
else:
    from google.colab import files
    up = files.upload()
    MODEL = list(up)[0]

n = sum(1 for l in open(MODEL, errors="ignore") if l.startswith(("ATOM", "HETATM")))
print("model:", MODEL, "-", n, "atom records")

## 3 · Measure the interfaces

This is the main calculation. It finds the symmetry, groups the chains into entities,
lists every contacting pair, computes buried surface atom by atom, and assigns the chemistry
of each contact.

`shape complementarity` is the slow option — it builds a solvent-excluded surface for every
interface. A small dimer takes a minute; a 30-chain assembly can take half an hour. Switch it
off for a first look.

`resolution` is the resolution of your map or diffraction data. It is quoted in every report
and sets how strict the distance criteria are. Leave it at 0 if you do not know it.


In [ ]:
#@title Run step 1 { display-mode: "form" }
resolution = 1.97  #@param {type:"number"}
shape_complementarity = True  #@param {type:"boolean"}
#@markdown Optional — name your proteins, e.g. `A,B=enzyme; C=inhibitor`
chain_names = ""  #@param {type:"string"}

cmd = "python 01_interface_analysis.py %s" % MODEL
if resolution and resolution > 0:
    cmd += " --resolution %g" % resolution
if shape_complementarity:
    cmd += " --shape --shape-workers 2"
if chain_names.strip():
    cmd += ' --names "%s"' % chain_names.strip()
print(cmd, "\n")
!{cmd}

## 4 · Derive the tables and the statistics

Secondary structure, burial inside the whole assembly, footprints per symmetry-independent
environment, energetics, packing, and the written report. No settings — it reads them back
from what step 3 wrote.


In [ ]:
#@title Run step 2 and show the report { display-mode: "form" }
!python 02_derive_tables.py {MODEL}

print("\n" + "=" * 78 + "\n")
print(open("stats_summary.txt", encoding="utf-8").read())

## 5 · Draw the figures


In [ ]:
#@title Run step 3 { display-mode: "form" }
residues_per_line = 350  #@param {type:"integer"}
#@markdown Optional — your own colours, e.g. `enzyme=#7FA8D4; inhibitor=#E8884A`
colours = ""  #@param {type:"string"}

cmd = "python 03_make_figures.py %s --per-line %d" % (MODEL, residues_per_line)
if colours.strip():
    cmd += ' --colours "%s"' % colours.strip()
print(cmd, "\n")
!{cmd}

In [ ]:
#@title Show the figures here { display-mode: "form" }
import glob
from IPython.display import Image, display, Markdown

figs = sorted(glob.glob("fig*.png"))
if not figs:
    print("no figures found - did step 3 run?")
for f in figs:
    display(Markdown("### " + f))
    display(Image(filename=f, width=950))

## 6 · Take everything with you

Every table is a plain CSV with a header row, so it opens in Excel, R or anything else.
The figures are written as both PNG and PDF, and the PDFs are vector — they can be edited
in Illustrator or Inkscape for a manuscript.


In [ ]:
#@title Download a zip of all results { display-mode: "form" }
import shutil, os
base = os.path.splitext(os.path.basename(MODEL))[0] + "_interface_analysis"
shutil.make_archive("/content/" + base, "zip", "/content/work")
print("packed", base + ".zip")
try:
    from google.colab import files
    files.download("/content/" + base + ".zip")
except Exception as e:
    print("not running in Colab - the file is at /content/%s.zip" % base)

---

### How to read three of the numbers

**Buried area** is computed pairwise in isolation and reported halved — the interface area,
not the sum of the two footprints. That is the same convention PISA uses, so the numbers are
directly comparable to it.

**Shape complementarity** (*S*<sub>c</sub>) here reads about **0.10 below** the original `sc`
program's scale: trypsin–BPTI gives 0.647 and the obligate histone-fold heterodimers of the
nucleosome give 0.598–0.620, against ~0.70 published. The class names on the figure are
Lawrence & Colman's, drawn on the original scale, so they read pessimistically. Use
*S*<sub>c</sub> to rank interfaces within one run; do not quote the class name without this
caveat.

**ΔG** is the Eisenberg & McLachlan atomic solvation term, fitted for the twenty amino acids
only. Nucleic acid and ligand atoms contribute buried area but nothing to ΔG. It is not a
binding free energy — there is no entropy and no electrostatics in it.

### If something goes wrong

- *Only one chain in the file* — there is no interface to measure. Some entries put the
  biological assembly in `REMARK 350` rather than in the coordinates; download the
  **biological assembly** file instead.
- *It is taking very long* — turn off shape complementarity in step 3, or raise
  `--shape-spacing` to 0.7.
- *The free Colab machine ran out of memory* on a very large assembly — run it locally, or on
  a cluster node; the code is identical and needs nothing but Python.
